## Advanced Indexing in PyTorch
(an added prelude to the coding task 1 from the 2024/25 exam)

### Indexing with a single array
Say we have a tensor `t` of shape (A, B, C).

For a flat array like `x = torch.tensor([5, 7, 7])`,<br>
`t[1, x, 2]` will give you a tensor of the same shape as `x`, namely `(3,)`, with values `[t[1, 5, 2], t[1, 7, 2], t[1, 7, 2]]`.

For a multi-dimensional array like `x = torch.tensor([[5, 6], [7, 8]])`, or $x = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$, <br>
`t[1, x, 2]` will give you a tensor of the same shape as x, namely `(2, 2)`, with values $\begin{bmatrix} t[1, 5, 2] & t[1, 6, 2] \\ t[1, 7, 2] & t[1, 8, 2] \end{bmatrix}$.

If `t` has more dimensions than used in the indexing, for example in `t[1, x]`, then again you get a tensor with values $\begin{bmatrix} t[1, 5] & t[1, 6] \\ t[1, 7] & t[1, 8] \end{bmatrix}$, but now those innermost values are not scalars, so the resulting tensor has shape (2, 2, C) here.

The same notation can also be used to modify selected values of a tensor. For example, `t[1, x, 2] = 0` will set the values of `t` at positions (1, 5, 2), (1, 6, 2), (1, 7, 2), and (1, 8, 2) to zero.

### Indexing with multiple arrays

You can index with multiple array, but they must have the same shape (or be broadcastable to the same shape).
The effect is that you access the elements of `t` at positions given by aligning those arrays, as if it were one array of tuples.

For flat arrays like `x = torch.tensor([1, 2])`, `y = torch.tensor([3, 4])`, `z = torch.tensor([5, 6])`,<br>
`t[x, y, z]` will give you a tensor of shape `(2,)` (same as those three arrays) with values `[t[1, 3, 5], t[2, 4, 6]]`.<br>
In other words, think of `t[x,y,z]` as indexing into t with tuples obtained by aligning the arrays x, y, z.

Note that `t[1, x, 2]` is equivalent to `t[torch.tensor([1, 1, 1]), x, torch.tensor([2, 2, 2])]`. So this is equivalent to broadcasting the constant indices to the shape of `x`.

For multi-dimensional arrays the result is entirely analogous. For example, for `x,y,z`, each of shape (M, N),<br>
`t[x, y, z]` will give you a tensor of shape (M, N) such that:<br>
`result[m, n] = t[x[m,n], y[m,n], z[m,n]]`, for all `(m, n)` in `range(M) × range(N)`.

As before, if `t` has more dimensions, for example in `t[x, y]`, then again you get a tensor with values `result[m, n] = t[x[m,n], y[m,n]]` which are not scalars, so the resulting tensor has shape (M, N, C) here.<br>
Equivalently `result[m, n, :] = t[x[m,n], y[m,n], :]`, for all `(m, n)`.

When assigning, the assigned value can also be a tensor of the same shape as the indexing arrays (or broadcastable).<br>
So `t[x, y, z] = w` will set `t[x[m,n], y[m,n], z[m,n]] = w[m,n]` for all `(m, n)`.

### A common example: one-hot encoding
In one-hot encoding, we start with `a = torch.zeros((B, C))` and a tensor of indices `tgt` of shape `(B,)` with values in `range(C)`.<br>
We want to set `a[b, tgt[b]] = 1` for all `b` in `range(B)`.<br>
So we write `a[?, tgt] = 1`, but what should we put in place of `?`? 

<details markdown="1">
<summary>Answer</summary>

`a[torch.arange(B), tgt] = 1` will do the trick, you index with two tensors of shape `(B,)`.

</details>

# Exam 2024/25 task 1: Patch pooling
Your task is to implement patch pooling. Patch pooling takes an input sequence $(a_0, a_1, \ldots, a_{B-1})$ of $D$-dimensional embeddings and outputs an output sequence $(b_0, \ldots, b_{k-1})$ of $D$-dimensional embeddings. The length of the output sequence is not longer than the length of the input sequence and is bounded by $P$. Each element of the input sequence is called a token. Each element of the output sequence is called a patch. Consecutive patches are constructed as a mean pooling of consecutive contiguous token spans.

You are given two tensors:
1. `batch` - a $3$-dimensional tensor, which is an input to a standard transformer model with the following dimensions:
* B - batch size
* S - sequence length
* D - dimension of embedding of a single token

`batch[x,y,:]` is the embedding of the $y$-th token of the $x$-th sequence in the `batch`.

2. `patch_lengths` - $2$-dimensional integer-valued tensor with the following dimensions:
* B - batch size
* P - maximal number of patches

`patch_lengths[x,y]` is the number of tokens forming patch number $y$ in the $x$-th sequence in the `batch`.

The output should be a $3$-dimensional tensor with batch of sequences of patch embeddings.

# Example
The following snippet
```python
batch = torch.tensor([[[ 1.,  1.,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  1.,  1.],
         [ 2.,  2.,  2.,  2.,  2.],
         [ 3.,  3.,  3.,  3.,  3.],
         [ 3.,  3.,  3.,  3.,  3.]],

        [[ 4.,  4.,  4.,  4.,  4.],
         [ 4.,  4.,  4.,  4.,  4.],
         [ 4.,  4.,  4.,  4.,  4.],
         [ 4.,  4.,  4.,  4.,  4.],
         [ 5.,  5.,  5.,  5.,  5.],
         [-1., -1., -1., -1., -1.]],

        [[ 6.,  6.,  6.,  6.,  6.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.]]])
patch_lengths = torch.tensor([[3, 1, 2],
        [4, 1, 0],
        [1, 0, 0]])
patch_pooling = PatchPooling()
output = patch_pooling(batch, patch_lengths)
output
```

should ouptut

```python
torch.tensor([[[1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3.]],

        [[4., 4., 4., 4., 4.],
         [5., 5., 5., 5., 5.],
         [-1., -1., -1., -1., -1.]],

        [[6., 6., 6., 6., 6.],
         [-1., -1., -1., -1., -1.],
         [-1., -1., -1., -1., -1.]]])
```

Remarks:

1. In this problem you can assume that embeddings of the padding token are vectors with all coordinates equal to $-1$.

2. Solutions will be graded with unit tests. You are given a single test case, which will be a part of evaluation.

3. Solutions not satisfying the below requirements will be graded up to 4pts:
* You are not allowed to call custom python functions
* You are not allowed to use Python loops
* Your are not allowed to use any other imports

#### Hints (not part of the original exam)
<details markdown="1">
<summary>Hint 1</summary>

First compute position of the start and end of each patch. What [operation](https://docs.pytorch.org/docs/2.14/generated/torch.cumsum.html) gives us that?

</details>

<details markdown="1">
<summary>Hint 2</summary>

Sums for arbitrary intervals can be quickly computed if you first use the same operation for the data `batch`.

</details>

<details markdown="1">
<summary>Hint 3</summary>

Compute prefix sums, then use multi-array indexing to access prefix sums that go until the end of each patch.
Then substract similar prefix sums that go until the start of each patch.

</details>

In [ ]:
%%file test_patch_pooling.py

import pytest
import torch


class PatchPooling(torch.nn.Module):
    def forward(self, batch: torch.Tensor, patch_lengths: torch.Tensor) -> torch.Tensor:
        B, S, D = batch.shape
        B_1, P = patch_lengths.shape

        assert B == B_1

        ### Your code goes here ###

        ###########################




class TestPatchPooling:
    @pytest.mark.parametrize(
        "batch,patch_lengths,expected_output",
        [
            (
                torch.tensor(
                    [
                        [
                            [1.0, 1.0, 1.0, 1.0, 1.0],
                            [1.0, 1.0, 1.0, 1.0, 1.0],
                            [1.0, 1.0, 1.0, 1.0, 1.0],
                            [2.0, 2.0, 2.0, 2.0, 2.0],
                            [3.0, 3.0, 3.0, 3.0, 3.0],
                            [3.0, 3.0, 3.0, 3.0, 3.0],
                        ],
                        [
                            [4.0, 4.0, 4.0, 4.0, 4.0],
                            [4.0, 4.0, 4.0, 4.0, 4.0],
                            [4.0, 4.0, 4.0, 4.0, 4.0],
                            [4.0, 4.0, 4.0, 4.0, 4.0],
                            [5.0, 5.0, 5.0, 5.0, 5.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                        ],
                        [
                            [6.0, 6.0, 6.0, 6.0, 6.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                        ],
                    ]
                ),
                torch.tensor([[3, 1, 2], [4, 1, 0], [1, 0, 0]]),
                torch.tensor(
                    [
                        [
                            [1.0, 1.0, 1.0, 1.0, 1.0],
                            [2.0, 2.0, 2.0, 2.0, 2.0],
                            [3.0, 3.0, 3.0, 3.0, 3.0],
                        ],
                        [
                            [4.0, 4.0, 4.0, 4.0, 4.0],
                            [5.0, 5.0, 5.0, 5.0, 5.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                        ],
                        [
                            [6.0, 6.0, 6.0, 6.0, 6.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                            [-1.0, -1.0, -1.0, -1.0, -1.0],
                        ],
                    ]
                ),
            )
        ],
    )
    def test_forward(
        self,
        batch: torch.Tensor,
        patch_lengths: torch.Tensor,
        expected_output: torch.Tensor,
    ) -> None:
        # given
        patch_pooling = PatchPooling()

        # when
        output = patch_pooling(batch=batch, patch_lengths=patch_lengths)

        # then
        assert torch.all(torch.isclose(output, expected_output))

Overwriting test_patch_pooling.py


In [28]:
!python -m pytest test_patch_pooling.py

======================================= test session starts ========================================
platform linux -- Python 3.11.11, pytest-8.3.4, pluggy-1.5.0
rootdir: /content
plugins: langsmith-0.3.2, typeguard-4.4.1, anyio-3.7.1
collected 1 item                                                                                   

test_patch_pooling.py .                                                                      [100%]

======================================== 1 passed in 1.90s =========================================
